In [1]:
import sqlite3
import pandas as pd

conn = sqlite3.connect(r"C:\Project_Data\e_commerce_olist\data\olist.db")

orders = pd.read_sql("SELECT order_id, customer_id, order_status, order_purchase_timestamp, order_delivered_carrier_date, order_delivered_customer_date, order_estimated_delivery_date FROM orders", conn, parse_dates=['order_purchase_timestamp','order_delivered_carrier_date','order_delivered_customer_date','order_estimated_delivery_date'])
order_items = pd.read_sql("SELECT order_id, shipping_limit_date FROM order_items", conn, parse_dates=['shipping_limit_date'])
order_reviews = pd.read_sql("SELECT order_id, review_score, review_creation_date FROM order_reviews", conn, parse_dates=['review_creation_date'])
customers = pd.read_sql("SELECT customer_id, customer_state FROM customers", conn)

# 1. orders_filtered
orders_filtered = orders[
    (orders['order_status'] == 'delivered') &
    (orders['order_delivered_customer_date'].notna()) &
    (orders['order_delivered_carrier_date'].notna()) &
    (orders['order_purchase_timestamp'] < orders['order_delivered_carrier_date']) &
    (orders['order_delivered_carrier_date'] < orders['order_delivered_customer_date'])
]

# 2. order_items_filtered
order_items_clean = order_items[order_items['order_id'] != 'c2bb89b5c1dd978d507284be78a04cb2']
order_items_filtered = order_items_clean.groupby('order_id', as_index=False)['shipping_limit_date'].min()
order_items_filtered.columns = ['order_id', 'shipping_date']

# 3. join_1 (inner join + filter SLA)
join_1 = orders_filtered.merge(order_items_filtered, on='order_id', how='inner')
join_1 = join_1[join_1['order_delivered_carrier_date'] <= join_1['shipping_date']]

# 4. order_reviews_filtered (ambil review terbaru per order_id)
order_reviews_sorted = order_reviews.sort_values('review_creation_date', ascending=False)
order_reviews_filtered = order_reviews_sorted.drop_duplicates(subset='order_id', keep='first')[['order_id','review_score']]

# 5. join_2 (left join + filter review_score not null)
join_2 = join_1.merge(order_reviews_filtered, on='order_id', how='left')
join_2 = join_2[join_2['review_score'].notna()]

# 6. join_3 (inner join customers)
mart = join_2.merge(customers, on='customer_id', how='inner')
mart = mart[['order_id','order_delivered_customer_date','order_estimated_delivery_date','review_score','customer_state']]

print(f"Total rows: {len(mart)}")
print(f"Duplicate order_id: {mart['order_id'].duplicated().sum()}")

# simpan balik ke database sebagai tabel mart
mart.to_sql('mart_delivery_performance', conn, if_exists='replace', index=False)
conn.close()

Total rows: 87027
Duplicate order_id: 0


In [2]:
print(f"Total Rows orders_filtered : {orders_filtered.shape[0]}")
print(f"Total Rows join_1 : {join_1.shape[0]}")
print(f"Total Rows join_2 : {join_2.shape[0]}")
print(f"Total Rows mart : {mart.shape[0]}")

Total Rows orders_filtered : 96272
Total Rows join_1 : 87570
Total Rows join_2 : 87027
Total Rows mart : 87027
